# Línea base de tarea única (modelo de Seminario I) — actividad 1.3

El `best.pt` original de Seminario I no sirve para comparar con el multitarea: se entrenó con el split
aleatorio de Roboflow y **vio 77 de las 108 fotos de nuestro test (37 de 39 frutos)**. Este notebook lo
reentrena sobre el split por fruto (852 / 232 / 108) y lo evalúa igual que al multitarea.

Se entrenan dos configuraciones (YOLOv8s-seg normal, clases palta y defecto, sin madurez):

| Config | Qué es | Qué responde |
|---|---|---|
| `replica_s1` | hiperparámetros exactos de Seminario I (mosaic 1.0, hsv 0.015/0.7/0.4) | qué tan bueno era de verdad el modelo de Seminario I |
| `ablacion` | los mismos hiperparámetros que `yolov8s_mt_final` | si el cabezal de madurez le quita calidad a los defectos |

**Uso:** Entorno de ejecución → Cambiar tipo → **GPU (T4)**, y luego *Ejecutar todo*.
Todo se guarda en el Drive (`SEMINARIO 1/SEMI2/linea_base_s1`): si Colab se desconecta, se vuelve a
ejecutar todo y el entrenamiento **continúa desde el último checkpoint**.
Tiempo estimado en T4: ~80 min por configuración (Seminario I tardó 81 min).

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

SALIDA = '/content/drive/MyDrive/SEMINARIO 1/SEMI2/linea_base_s1'   # cambiar si la carpeta está en otro lado
CONFIGS = ['replica_s1', 'ablacion']                                   # quitar una para entrenar solo la otra
import os; os.makedirs(SALIDA, exist_ok=True)

## 1. Código, dependencias y datos (repo y release públicos de GitHub)

In [ ]:
%cd /content
!test -d SEMINARIO2 || git clone -q https://github.com/fvm14/SEMINARIO2.git
%cd /content/SEMINARIO2
!git pull -q
!pip -q install ultralytics==8.4.160
!test -f imagenes_anotadas.zip || wget -q https://github.com/fvm14/SEMINARIO2/releases/download/v0.2-etapa2/imagenes_anotadas.zip
!unzip -q -n imagenes_anotadas.zip -d data/raw
!python scripts/prepare_yolo.py | tail -4

## 2. Entrenamiento (se reanuda solo si se cortó)

In [ ]:
import os, subprocess

def pesos(cfg, nombre='best.pt'):
    return f'{SALIDA}/yolov8s_{cfg}/weights/{nombre}'

for cfg in CONFIGS:
    marca = f'{SALIDA}/yolov8s_{cfg}/terminado.txt'
    if os.path.exists(marca):
        print(cfg, '-> ya entrenado'); continue
    if os.path.exists(pesos(cfg, 'last.pt')):
        cmd = ['python', 'scripts/train_linea_base_s1.py', '--config', cfg, '--resume', pesos(cfg, 'last.pt')]
    else:
        cmd = ['python', 'scripts/train_linea_base_s1.py', '--config', cfg, '--project', SALIDA]
    print(' '.join(cmd)); subprocess.run(cmd, check=True)
    open(marca, 'w').write('ok')

## 3. Calibración del umbral de defecto en validación (nunca en test)

In [ ]:
import pandas as pd
umbral = {}
for cfg in CONFIGS:
    csv = f'{SALIDA}/yolov8s_{cfg}/calibracion_defecto_val.csv'
    if not os.path.exists(csv.replace('.csv', '_resumen.csv')):
        subprocess.run(['python', 'scripts/calibrar_umbral_defecto.py', '--weights', pesos(cfg), '--salida', csv], check=True)
    r = pd.read_csv(csv.replace('.csv', '_resumen.csv'))
    umbral[cfg] = float(r.sort_values(['iou_defecto', 'acc_ocde'], ascending=False).iloc[0].umbral)
    print(cfg, '-> umbral de defecto', umbral[cfg])

## 4. Evaluación en test (108 fotos, 39 frutos)

In [ ]:
for cfg in CONFIGS:
    subprocess.run(['python', 'scripts/eval_linea_base_s1.py', '--weights', pesos(cfg),
                    '--conf-defecto', str(umbral[cfg]), '--out-dir', f'{SALIDA}/yolov8s_{cfg}'], check=True)

## 5. Bootstrap pareado por fruto contra el multitarea

A = línea base, B = multitarea. Solo leer las filas de **OCDE, recall de Rechazado y MAE del ratio**:
la línea base no predice madurez (sus columnas de madurez se llenan con el nombre del archivo).

In [ ]:
MT = {'mt_final (852 img)': 'resultados/yolo_multitarea/yolov8s_mt_final/eval_test_20260923_211752',
      'mt_ronda1 (5,475 img)': 'resultados/yolo_multitarea/yolov8s_mt_ronda1/eval_test_20260924_021049'}
for cfg in CONFIGS:
    for nombre, ruta in MT.items():
        print(f'\n===== {cfg}  vs  {nombre} =====')
        out = subprocess.run(['python', 'scripts/bootstrap_ic.py', '--eval', f'{SALIDA}/yolov8s_{cfg}/eval_test',
                              '--comparar', ruta], capture_output=True, text=True).stdout
        print(out)
        open(f'{SALIDA}/yolov8s_{cfg}/bootstrap_vs_{nombre.split()[0]}.txt', 'w').write(out)

## 6. Resumen

In [ ]:
import json
def fila(nombre, m):
    u = m['ultralytics']['global']; px = m['pixel_con_roi']; o = m['ocde']
    return {'modelo': nombre, 'P(M)': u.get('metrics/precision(M)'), 'R(M)': u.get('metrics/recall(M)'),
            'mAP50(M)': u.get('metrics/mAP50(M)'), 'mAP50-95(M)': u.get('metrics/mAP50-95(M)'),
            'IoU defecto': px.get('iou_defecto'), 'acierto OCDE': o.get('accuracy'),
            'recall Rechazado': o.get('recall_rechazado'), 'MAE ratio': o.get('mae_ratio'), 'umbral': m.get('conf_defecto')}
filas = [fila(cfg, json.load(open(f'{SALIDA}/yolov8s_{cfg}/eval_test/metricas.json'))) for cfg in CONFIGS]
tabla = pd.DataFrame(filas)
tabla.to_csv(f'{SALIDA}/resumen_linea_base.csv', index=False)
tabla